In [0]:
import os
import json

is_job = False
try:
    raw = dbutils.jobs.taskValues.get("join_customer_and_sales_data", "metadata", None)
    print("Notebook is running in Job mode")
    print(f"raw ---> {raw}")

    # get data from the json output of the job ingest_customers_data
    meta = json.loads(raw)
    catalog          = meta.get("catalog")
    schema           = meta.get("schema")
    table_name       = meta.get("table_name")
    gold_table_name = meta.get("gold_table_name")
    is_job = True
except:
    is_job = False
    

In [0]:
if is_job:
    spark.sql(f"""
            CREATE OR REPLACE TABLE {catalog}.{schema}.{gold_table_name} AS
                SELECT
                    customer_id,
                    first_name,
                    last_name,
                    phone,
                    sale_id,
                    -- Convert string to DATE
                    TO_DATE(sale_date, 'yyyy-MM-dd') AS sale_date,
                    -- Convert string to DOUBLE
                    CAST(sales_amount AS DOUBLE) AS sales_amount,
                    payment_method,
                    state
                FROM {catalog}.{schema}.{table_name};
            """)
else:
    print(f"Notebook is running in interactive mode skipping data cleaning process!")